<a href="https://colab.research.google.com/github/adeleke1508/flyrank_week_one_ml_project_1/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adeleke1508/flyrank_week_one_ml_project_1/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os
import pandas as pd
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)

df = pd.read_csv('/content/drive/MyDrive/merged_and_cleaned_dim_content_and_fact_content_daily_performance_month_2026_05.csv')

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The action playbook prioritizes content using an impression-weighted CTR gap ($\text{impressions} \times \text{ctr\_gap}$), benchmarking each page against expected performance for its search position bucket.

* **Primary Action (`REFRESH`):** Applied to mature content (`old = 1`, created $\le$ March 15, 2026) that exhibits a measurable deficit in click-through rate relative to its position norm (`ctr_gap > 0`).
* **Reason Code (`STALE_BELOW_POSITION_CTR`):** Indicates that high impression volume combined with sub-benchmark CTR points to content staleness, outdated snippets, or mismatched search intent.
* **Secondary Action (`NONE`):** Assigned to fresh or well-performing pages where current CTR meets or exceeds positional expectations.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

df['old'] = np.where(df['content_created_date'] <= '2026-03-15', 1, 0)
conditions = (df['old'] == 1) & (df['content_updated_date'] <= '2026-06-13')
df['old_but_updated_recently'] = np.where(conditions, 0, 1)

d = df[df['impressions'] >= 100].copy()
d['pos_x_imp'] = d['avg_position'] * d['impressions']

pages = d.groupby('content_hash_id').agg(
    client=('client', 'first'),
    content_type=('content_type', 'first'),
    old=('old', 'first'),
    old_but_updated_recently=('old_but_updated_recently', 'first'),
    impressions=('impressions', 'sum'),
    clicks=('clicks', 'sum'),
    pos_x_imp=('pos_x_imp', 'sum'),
    n_reports=('days_with_gsc', 'nunique'),
).reset_index()

pages['avg_position'] = pages['pos_x_imp'] / pages['impressions']
pages['ctr'] = pages['clicks'] / pages['impressions']
pages = pages[pages['impressions'] >= 100].copy()

pages['pos_bucket'] = pd.cut(pages['avg_position'], [0, 3, 5, 7, 10, 20, 50, 100])
g = pages.groupby('pos_bucket', observed=True)
pages['expected_ctr'] = g['clicks'].transform('sum') / g['impressions'].transform('sum')
pages['ctr_gap'] = (pages['expected_ctr'] - pages['ctr']).clip(lower=0)

flag = (pages['old'] == 1) & (pages['old_but_updated_recently'] == 0) & (pages['ctr_gap'] > 0)
pages['score'] = np.where(flag, pages['impressions'] * pages['ctr_gap'], 0)
pages['reason_code'] = np.where(flag, 'STALE_BELOW_POSITION_CTR', '')
pages['action'] = np.where(flag, 'REFRESH', 'NONE')

queue = pages[flag].sort_values('score', ascending=False)
print("Total actionable items in queue:", len(queue))
queue.head(10)


Total actionable items in queue: 17512


,content_hash_id,client,content_type,old,old_but_updated_recently,impressions,clicks,pos_x_imp,n_reports,avg_position,ctr,pos_bucket,expected_ctr,ctr_gap,score,reason_code,action
15898,content_39e19a3ec2d95f9d,client_1a730cb2640a1abf,keyword article,1,0,189298,2,1908139.0,1,10.080080,0.000011,"(10, 20]",0.004169,0.004159,787.252090,STALE_BELOW_POSITION_CTR,REFRESH
52443,content_bddfdd871aa09fbe,client_23a62021009f63c4,keyword article,1,0,112353,4,131913.0,1,1.174094,0.000036,"(0, 3]",0.005534,0.005498,617.771550,STALE_BELOW_POSITION_CTR,REFRESH
27976,content_65c75874a23fca87,client_23a62021009f63c4,keyword article,1,0,191566,32,1666112.0,1,8.697326,0.000167,"(7, 10]",0.003313,0.003146,602.715633,STALE_BELOW_POSITION_CTR,REFRESH
5793,content_15168cd1c94e3529,client_23a62021009f63c4,keyword article,1,0,151562,332,513995.0,1,3.391318,0.002191,"(3, 5]",0.005959,0.003768,571.135348,STALE_BELOW_POSITION_CTR,REFRESH
28056,content_661a7734f691bef5,client_23a62021009f63c4,keyword article,1,0,188308,253,3704157.0,1,19.670736,0.001344,"(10, 20]",0.004169,0.002826,532.124421,STALE_BELOW_POSITION_CTR,REFRESH
24644,content_599eba17030c875c,client_a80fca3f171ed1de,keyword article,1,0,145499,130,794318.0,1,5.459268,0.000893,"(5, 7]",0.004179,0.003286,478.080074,STALE_BELOW_POSITION_CTR,REFRESH
18957,content_44f34c0a90047651,client_23a62021009f63c4,keyword article,1,0,136735,35,1213590.0,1,8.875489,0.000256,"(7, 10]",0.003313,0.003057,418.044079,STALE_BELOW_POSITION_CTR,REFRESH
53574,content_c1f764a2f362d1c3,client_a80fca3f171ed1de,keyword article,1,0,118966,16,924441.0,1,7.770632,0.000134,"(7, 10]",0.003313,0.003179,378.170051,STALE_BELOW_POSITION_CTR,REFRESH
47435,content_abe4335206065315,client_a80fca3f171ed1de,keyword article,1,0,97111,68,654233.0,1,6.736961,0.000700,"(5, 7]",0.004179,0.003479,337.853401,STALE_BELOW_POSITION_CTR,REFRESH
28623,content_684037d084cbe6cc,client_a80fca3f171ed1de,keyword article,1,0,97236,91,494655.0,1,5.087159,0.000936,"(5, 7]",0.004179,0.003243,315.375810,STALE_BELOW_POSITION_CTR,REFRESH


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

* **Intended Use:** Designed for editorial and SEO operations teams to triage large catalogs of published articles, identifying high-volume pages where organic traffic is suppressed by outdated content or sub-optimal title tags/snippets.
* **Operational Limits:**
  1. **Position Dependency:** The queue relies heavily on average search position. Pages experiencing high position volatility over the monthly window may produce distorted CTR gaps.
  2. **SERP Layout Bias:** High-impression pages with zero clicks can sometimes reflect dominant AI Overviews or Featured Snippets absorbing traffic rather than simple content staleness.
  3. **Client Concentration:** Global pooling benchmarks can disproportionately highlight clients with larger aggregate search volumes.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(queue['client'].value_counts().head(5))


client
client_23a62021009f63c4    4598
client_73cda7b4e4f265ea    1963
client_20259bd6705d81d4    1949
client_a80fca3f171ed1de    1473
client_3f0ce4d44fe94f3d    1162
Name: count, dtype: int64


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

* **Mandatory Human Review Rules:** Before executing a content refresh on any flagged URL, an editor must verify:
  1. Canonical tag health and proper internal redirects.
  2. Whether a recent manual SERP check reveals competing features (e.g., video carousels or ads) suppressing click-throughs.
  3. Alignment between the primary keyword intent and the current article subheadings.
* **What Should NEVER Be Automated (The No-Go List):**
  * Automated mass publishing or rewriting of content without human editorial oversight.
  * Automated de-indexing or deletion of pages solely based on low score thresholds.
  * Blind programmatic changes to title tags without checking brand guidelines or tone of voice.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print(queue[['impressions', 'clicks', 'avg_position', 'ctr_gap', 'score']].describe().round(4))

       impressions      clicks  avg_position     ctr_gap       score
count   17512.0000  17512.0000    17512.0000  17512.0000  17512.0000
mean     2252.7318      3.6290       22.0811      0.0023      4.4103
std      7060.9024     15.1552       15.0212      0.0013     18.8952
min       100.0000      0.0000        0.2126      0.0000      0.0002
25%       214.0000      0.0000       10.0991      0.0012      0.4482
50%       503.0000      0.0000       18.1664      0.0022      0.9374
75%      1522.0000      2.0000       30.7175      0.0033      2.5407
max    202315.0000    742.0000       98.8457      0.0060    787.2521


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

* **Staleness Triggers:** The recommendations degrade if organic click-distribution shifts significantly across position brackets or if major Google core algorithm updates alter baseline CTR curves.
* **Retrain & Recalibration Cadence:**
  * **Monthly:** Re-run the aggregation pipeline on new monthly GSC performance dumps to update positional benchmarks.
  * **Performance Drift Check:** Monitor the mean Absolute Error (MAE) of the baseline position model. If test MAE degrades by >15% compared to historical baselines, recalibrate the position buckets or inspect for tracking code failures.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print(queue.groupby('pos_bucket', observed=True)['score'].agg(['count', 'mean', 'max']).round(2))

            count   mean     max
pos_bucket                      
(0, 3]        104  19.69  617.77
(3, 5]        451  19.32  571.14
(5, 7]       1187  14.19  478.08
(7, 10]      2562   6.66  602.72
(10, 20]     5321   3.37  787.25
(20, 50]     6975   2.05  132.22
(50, 100]     912   0.34   60.92


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

* **Artifact Export:** Exports the final ranked action queue to `work/outputs/baseline_action_score.csv` and saves summary diagnostic figures to `work/figures/` to serve as the empirical foundation for the research paper's recommendations section.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import matplotlib.pyplot as plt
import seaborn as sns

output_path = 'work/outputs/baseline_action_score.csv'
queue.to_csv(output_path, index=False)
print(f"Queue successfully exported to {output_path} ({len(queue)} rows)")

plt.figure(figsize=(8, 5))
sns.scatterplot(data=queue.head(1000), x='avg_position', y='ctr_gap', hue='impressions', palette='viridis', alpha=0.6)
plt.title('Top Action Queue: CTR Gap vs. Average Position')
plt.xlabel('Average Search Position')
plt.ylabel('CTR Gap (Expected - Actual)')
plt.savefig('work/figures/ctr_gap_vs_position.png', dpi=300, bbox_inches='tight')
plt.close()
print("Figure saved to work/figures/ctr_gap_vs_position.png")

Queue successfully exported to work/outputs/baseline_action_score.csv (17512 rows)
Figure saved to work/figures/ctr_gap_vs_position.png


6. Showcase Demo Outline (5 Minutes)

* **The Question (1 min)**: How can we predict search query performance and identify stale content needing immediate refresh across 79 million rows of production search data without overfitting or data leakage?
* **The Method (1 min)**: Built a time-aware validation split (chronological cutoff by creation date) and combined a position-decay baseline with a 9-feature KMeans structural clustering lane (`word_count`, `char_count`, `keyword_token_count`, `url_char_count`, `category_count`, `backlinks`, `search_volume`, `competition`, `cpc`).
* **One Key Chart / Result (1 min)**: Demonstrated that Click-Through Rate (CTR) falls monotonically as average search position worsens (mean CTR dropping from 0.0062 for positions 1–3 to 0.0007 for positions 51–100, with zero-click share rising from 22% to 80%).

7. Shareable Cuts

A. Short Social Post (Methodology & Journey)
> "Excited to share my latest machine learning capstone analyzing search query performance on production-scale data! I built a time-aware validation pipeline using 9 structural features and KMeans clustering to benchmark content click-through rates against position decay. The biggest takeaway? While structural archetypes offer marginal gains, client-level behavioral variance is massive. Check out the full research paper and reproducible notebooks on my repo!"

B. Employer-Facing Summary (3 Sentences)
> "I engineered a production-grade search performance model on 79 million rows of real data to predict click-through rates and prioritize content refreshes. By implementing time-aware train/test splits and structural KMeans clustering, I quantified position-decay benchmarks and identified high-impact optimization targets. The resulting decision-support playbook helps teams systematically recover missed clicks while maintaining strict data governance."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.